# TeamFlow message sentiment: Naive Bayes

This notebook loads the TweetEval short-text sentiment splits, normalizes URLs and mentions, trains a TF-IDF + Multinomial Naive Bayes classifier, and reports validation and held-out test metrics. TweetEval is a tweet corpus; treat its results as a baseline rather than a workplace-chat accuracy claim.

## 1. Load and inspect the labeled splits

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / 'ML' if (Path.cwd() / 'ML').is_dir() else Path.cwd()))
from train_sentiment import CLASS_ORDER, MODEL_PATH, build_pipeline, evaluate, load_split

train_text, train_labels = load_split('train')
val_text, val_labels = load_split('val')
test_text, test_labels = load_split('test')
print(f'train={len(train_text):,}, validation={len(val_text):,}, test={len(test_text):,}')
print('classes:', CLASS_ORDER)
print('sample:', train_text[0], '->', train_labels[0])

## 2. Preprocess and train
The pipeline replaces links and user mentions with tokens, then learns word unigram/bigram TF-IDF features and class probabilities with Multinomial Naive Bayes.

In [ ]:
validation_model = build_pipeline().fit(train_text, train_labels)
validation_metrics = evaluate(validation_model, val_text, val_labels)
print(f'Validation accuracy: {validation_metrics["accuracy"]:.4f}')

## 3. Final fit and held-out evaluation
After validation, fit on train + validation and evaluate once on the separate test split. The confusion matrix order is negative, neutral, positive.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import joblib

final_model = build_pipeline().fit(train_text + val_text, train_labels + val_labels)
test_metrics = evaluate(final_model, test_text, test_labels)
print(f'Test accuracy: {test_metrics["accuracy"]:.4f}')
print(classification_report(test_labels, final_model.predict(test_text), labels=CLASS_ORDER, zero_division=0))
print('Confusion matrix:\n', confusion_matrix(test_labels, final_model.predict(test_text), labels=CLASS_ORDER))
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, MODEL_PATH, compress=3)
print('Saved:', MODEL_PATH)